In [ ]:
import os
import sys
import json
import random
import datetime
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

warnings.simplefilter(action="ignore", category=FutureWarning)

sys.path.append("..")

from models.buffers import NaiveRehearsalBuffer
from models.heuristicSplitModel import DualBranchModel, DualBranchModel_fusions
from models.training_utils import (
    heuristic_dualbranch_batch,
    unified_train_loop,
    archive_domain_models,
)
from data_processing.data_utils import (
    get_transform,
    serialize_transform,
    get_domain_dataloaders,
    get_domain_dataloaders_from_hdf5,
    get_crossvalidation_domain_loaders,
    pool_domain_dataloaders,
    IMAGENET_NORM,
)

In [ ]:
LABEL_COLS = [
    "Vaccum Cleaning", "Mopping the Floor", "Carry Warm Food",
    "Carry Cold Food", "Carry Drinks", "Carry Small Objects",
    "Carry Large Objects", "Cleaning", "Starting a conversation",
]

DEFAULT_DOMAINS = ["Home", "BigOffice-2", "BigOffice-3", "Hallway", "MeetingRoom", "SmallOffice"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_DIR = "../checkpoints"

GLOBAL_SEED = 42

NUM_FOLDS = 1

TRAIN_CONFIG = {
    "seed": GLOBAL_SEED,
    "epochs": 60,
    "batch_sizes": (32, 64, 64),        # (train, val, test)
    "learning_rate": 1e-3,
    "dropout_rate": 0.3,
    "branch_norm": True,
    "gradient_clipping": True,
    "early_stopping": True,
    "early_stopping_patience": 15,
    "early_stopping_delta": 1e-3,
    "num_workers": 0,
    "pin_memory": True,
    "persistent_workers": False,
    "scheduler_fn": get_linear_schedule_with_warmup,
    "scheduler_warmup": 0.05,
    "refresh_optimiser": False,
    "checkpoint_dir": CHECKPOINT_DIR,
    "validation_set": "val",
}

### Helpers

In [ ]:
def _scenario(
        ablation="base", 
        hdf5_path="../data/mean_data_pepper.hdf5", 
        transform_list=[get_transform((144, 256), IMAGENET_NORM)] * 2, 
        img_path_cols=["image_path_env_boxes_plus", "image_path_soc"], 
        joint_training=False, 
        use_buffer=True, 
        domains=DEFAULT_DOMAINS
        ):
    """Shared defaults for the scenarios."""
    return {
        "ablation": ablation,
        "buffer_size": 60,
        "train_val_path": None,
        "test_path": None,
        "hdf5_path": hdf5_path,
        "transform_list": transform_list,
        "img_path_cols": img_path_cols,
        "scene_as_label": False,
        "loss_fn": nn.MSELoss(),
        "setup": {"branch": "mobilenetv2"},
        "joint_training": joint_training,
        "use_buffer": use_buffer,
        "domains": domains,
    }

def set_seed(seed: int) -> None:
    """Set all relevant random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def resolve_ablation_setup(ablation: str, scene_as_label: bool, base_setup: dict) -> dict:
    """Translate the ablation name into the model 'setup' dict."""
    setup = dict(base_setup)
    if ablation == "nomask":
        setup["ablation"] = "focus"
    elif ablation == "onlysoc":
        setup["ablation"] = "scene"
    elif ablation == "onlyenv":
        setup["ablation"] = "focus"
    else:
        setup["ablation"] = False

    if scene_as_label:
        setup["scene"] = "label"
    return setup


def build_dataloaders(scenario: dict, hdf5_dataset_path: str, cfg: dict):
    """Build (and optionally pool) the domain dataloaders for one fold."""
    if scenario["hdf5_path"]:
        domain_dataloaders = get_domain_dataloaders_from_hdf5(
            hdf5_path=hdf5_dataset_path,
            domains=scenario["domains"],
            img_path_cols=scenario["img_path_cols"],
            num_workers=cfg["num_workers"],
            set_first_element_as_domain_label=scenario["scene_as_label"],
            pin_memory=cfg["pin_memory"],
            persistent_workers=cfg["persistent_workers"],
        )
    else:
        train_df = pd.read_pickle(scenario["train_val_path"]) if scenario["train_val_path"] else None
        test_df = pd.read_pickle(scenario["test_path"]) if scenario["test_path"] else None
        domain_dataloaders = get_domain_dataloaders(
            train_df,
            seed=cfg["seed"],
            batch_sizes=cfg["batch_sizes"],
            img_path_cols=scenario["img_path_cols"],
            transforms=scenario["transform_list"],
            num_workers=cfg["num_workers"],
            include_test=test_df,
            set_first_element_as_domain_label=scenario["scene_as_label"],
            pin_memory=cfg["pin_memory"],
            persistent_workers=cfg["persistent_workers"],
        )

    if scenario["joint_training"]:
        domain_dataloaders = pool_domain_dataloaders(domain_dataloaders)

    return domain_dataloaders


def build_run_config(name, scenario, hdf5_dataset_path, model, buffer, optimizer,
                      setup, exp_name, dualbranch_kwargs, cfg):
    """Assemble the per-run config dict logged alongside checkpoints."""
    return {
        "timestamp": datetime.datetime.now().strftime("%Y%m%d_%H%M%S"),
        "model": {
            "name": str(type(model)),
            "setup": setup,
            "branch_norm": cfg["branch_norm"],
            "dropout_rate": cfg["dropout_rate"],
        },
        "buffer": str(type(buffer)),
        "buffer_size": scenario["buffer_size"],
        "optimizer": str(type(optimizer)),
        "learning_rate": cfg["learning_rate"],
        "scheduler": {
            "type": "transformers.get_linear_schedule_with_warmup",
            "warmup": cfg["scheduler_warmup"],
        },
        "device": str(DEVICE),
        "loss": str(dualbranch_kwargs["mse_criterion"]),
        "joint_training": scenario["joint_training"],
        "seed": cfg["seed"],
        "domains_order": scenario["domains"],
        "train_val_df": scenario["train_val_path"],
        "test_df": scenario["test_path"],
        "hdf5_dataset_path": hdf5_dataset_path,
        "input_columns": scenario["img_path_cols"],
        "input_transforms": [serialize_transform(t) for t in scenario["transform_list"]],
        "dataloader": {
            "batch_size": cfg["batch_sizes"],
            "num_workers": cfg["num_workers"],
            "pin_memory": cfg["pin_memory"],
            "persistent_workers": cfg["persistent_workers"],
        },
        "epochs": cfg["epochs"],
        "early_stopping": {
            "patience": cfg["early_stopping_patience"],
            "delta": cfg["early_stopping_delta"],
        },
        "gradient_clipping": cfg["gradient_clipping"],
        "refresh_optimiser": cfg["refresh_optimiser"],
    }


In [ ]:

def run_scenario(name: str, scenario: dict, cfg: dict) -> None:
    """
    Run 5-fold cross-validation for a single scenario.

    Note: set_seed is called once per scenario (before the fold loop),
    matching the original script's behaviour exactly - it is NOT reset
    at the start of every fold.
    """
    set_seed(cfg["seed"])

    base_hdf5_path = scenario["hdf5_path"]

    for fold in range(NUM_FOLDS):
        print(f"\nTesting fold: {fold}")

        hdf5_dataset_path = (
            base_hdf5_path[:-5] + f"_fold{fold}.hdf5" if base_hdf5_path else None
        )

        domain_dataloaders = build_dataloaders(scenario, hdf5_dataset_path, cfg)

        print(f"\nTesting: {name} Ablation: {scenario['ablation']}")

        setup = resolve_ablation_setup(
            scenario["ablation"], scenario["scene_as_label"], scenario["setup"]
        )

        model = DualBranchModel(
            dropout_rate=cfg["dropout_rate"], setup=setup, branch_norm=cfg["branch_norm"]
        )
        dual_model = model.to(DEVICE)

        trainable_params = [p for p in dual_model.parameters() if p.requires_grad]
        optimizer = torch.optim.Adam(trainable_params, lr=cfg["learning_rate"])

        buffer = (
            NaiveRehearsalBuffer(buffer_size=scenario["buffer_size"])
            if scenario["use_buffer"] else None
        )

        exp_name = (
            f"{name}_fold={fold}_buffer_size={scenario['buffer_size']}"
            f"_epochs={cfg['epochs']}_seed={cfg['seed']}"
            f"_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
        )

        scheduler = (cfg["scheduler_fn"], cfg["scheduler_warmup"])
        dualbranch_kwargs = {"mse_criterion": scenario["loss_fn"]}

        run_config = build_run_config(
            name, scenario, hdf5_dataset_path, model, buffer, optimizer,
            setup, exp_name, dualbranch_kwargs, cfg,
        )
        with open(f"{cfg['checkpoint_dir']}/{exp_name}_config.json", "w") as f:
            json.dump(run_config, f, indent=4)

        unified_train_loop(
            model=dual_model,
            domains=scenario["domains"] if not scenario["joint_training"] else ["joint"],
            domain_dataloaders=domain_dataloaders,
            buffer=buffer,
            optimizer=optimizer,
            device=DEVICE,
            batch_fn=heuristic_dualbranch_batch,
            batch_kwargs=dualbranch_kwargs,
            num_epochs=cfg["epochs"],
            exp_name=exp_name,
            gradient_clipping=cfg["gradient_clipping"],
            checkpoint_dir=cfg["checkpoint_dir"],
            validation_set=cfg["validation_set"],
            scheduler=scheduler,
            refresh_optimiser=cfg["refresh_optimiser"],
            early_stopping=cfg["early_stopping"],
        )


### Train

In [ ]:

SCENARIOS = {
    "dual_branch.base": _scenario(),
    "single_branch.base": _scenario(
        ablation='nomask',
        transform_list=[get_transform((144,256), IMAGENET_NORM)],
        img_path_cols=['image_path'],
        ),
    "dual_branch_ft.base": _scenario(
        joint_training=True,
        use_buffer=False
        ),
    "single_branch_ft.base": _scenario(
        ablation='nomask',
        transform_list=[get_transform((144,256), IMAGENET_NORM)],
        img_path_cols=['image_path'],
        joint_training=True,
        use_buffer=False,
        ),
    "silh.base": _scenario(
        img_path_cols=["image_path_env", "image_path_soc"]
        ),
    "closeup.base": _scenario(
        hdf5_path="../data/robotfocus128_pepper_data.hdf5",
        transform_list=[get_transform((144, 256), IMAGENET_NORM), get_transform((128, 128), IMAGENET_NORM)],
        img_path_cols=["image_path_scene", "image_path_focus_x2"],
        ),
    "order_complexup.base": _scenario(domains=["Hallway", "SmallOffice", "MeetingRoom", "BigOffice-3", "BigOffice-2", "Home"]),
    "order_complexdown.base": _scenario(domains=["Home", "BigOffice-2", "BigOffice-3", "MeetingRoom", "SmallOffice", "Hallway"]),
    "order_contrA.base": _scenario(domains=["BigOffice-3", "SmallOffice", "Home", "Hallway", "MeetingRoom", "BigOffice-2"]),
    "order_contrB.base": _scenario(domains=["MeetingRoom", "Home", "Hallway", "BigOffice-2", "SmallOffice", "BigOffice-3"]),
}

In [ ]:
def main():
    # Global seed set once at import time, matching the original script.
    set_seed(GLOBAL_SEED)

    os.makedirs(TRAIN_CONFIG["checkpoint_dir"], exist_ok=True)

    for name, scenario in SCENARIOS.items():
        run_scenario(name, scenario, TRAIN_CONFIG)

    print("=" * 80)
    print("All training completed!")
    print("=" * 80)


if __name__ == "__main__":
    main()

### Legacy

In [ ]:
import pandas as pd
import torchvision.transforms as T
import numpy as np
import random
import torch
import torch.nn as nn
import datetime
import time
import os
import json
from transformers import get_linear_schedule_with_warmup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True  # Enforce deterministic algorithms
        torch.backends.cudnn.benchmark = False     # Disable benchmark for reproducibility

    os.environ['PYTHONHASHSEED'] = str(seed)       # Seed Python hashing, which can affect ordering
set_seed(42)


LABEL_COLS = [
    "Vaccum Cleaning", "Mopping the Floor", "Carry Warm Food",
    "Carry Cold Food", "Carry Drinks", "Carry Small Objects",
    "Carry Large Objects", "Cleaning", "Starting a conversation"
]


import sys
sys.path.append('..')

from models.buffers import NaiveRehearsalBuffer
from data_processing.data_utils import get_transform, serialize_transform, get_domain_dataloaders, pool_domain_dataloaders, pool_domain_dataloaders, get_crossvalidation_domain_loaders, IMAGENET_NORM, get_domain_dataloaders_from_hdf5
from models.training_utils import heuristic_dualbranch_batch, unified_train_loop, archive_domain_models
from models.heuristicSplitModel import DualBranchModel, DualBranchModel_fusions

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    

default_domains = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']

testing_scenarios = {
    'single_ft_b60.nomask':   ('nomask', 60, 120, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)],     ['image_path'],                                     False, nn.MSELoss(), {'branch':'mobilenetv2', 'ablation':'focus'},  True, False, default_domains),
    'default_ft_b60.base':    ('base', 60, 120, None, None,   "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      True, False, default_domains),
    'default_b60.base':       ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, default_domains),
    'single_b60.nomask':      ('nomask', 60, 60, None, None,  "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)],     ['image_path'],                                     False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, default_domains),
    'silh.base':              ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5",  [get_transform((144,256), IMAGENET_NORM)]*2,  ['image_path_env', 'image_path_soc'],               False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, default_domains),
    'closeup.base':           ('base', 60, 60, None, None,    "../data/robotfocus128_pepper_data.hdf5", [get_transform((144,256), IMAGENET_NORM), get_transform((128,128), IMAGENET_NORM)], ['image_path_scene', 'image_path_focus_x2'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, default_domains),
    'order_complexup.base':   ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, ['Hallway', 'SmallOffice', 'MeetingRoom', 'BigOffice-3', 'BigOffice-2', 'Home']),
    'order_complexdown.base': ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, ['Home', 'BigOffice-2', 'BigOffice-3', 'MeetingRoom', 'SmallOffice', 'Hallway']),
    'order_contrA.base':      ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, ['BigOffice-3', 'SmallOffice', 'Home', 'Hallway', 'MeetingRoom', 'BigOffice-2']),
    'order_contrB.base':      ('base', 60, 60, None, None,    "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2,   ['image_path_env_boxes_plus', 'image_path_soc'],    False, nn.MSELoss(), {'branch':'mobilenetv2'},                      False, True, ['MeetingRoom', 'Home', 'Hallway', 'BigOffice-2', 'SmallOffice', 'BigOffice-3']),
}
    
for name, (ablation, epochs, buffer_size, train_val_path, test_path, hdf5_path, transform_list, img_path_cols, scene_as_label, loss_fn, setup, joint_training, use_buffer, domains) in testing_scenarios.items():
        train_df = pd.read_pickle(train_val_path) if train_val_path is not None else None
        test_df = pd.read_pickle(test_path) if test_path is not None else None
        main_hdf5_dataset_path = hdf5_path
        setup=setup #{'branch':'mobilenetv2'}
        seed=42
        set_seed(seed)
        grad_clipping=True
        batch=(32, 64, 64)
        buffer_size=buffer_size
        branch_norm=True
        earlystopping=True
        epochs=epochs
        joint_training=joint_training
        scene_as_label=scene_as_label
        pin_memory=True
        persistent_workers=False
        num_workers=0
        use_buffer=use_buffer


        for fold in range(0,5):
            # if fold != 0:
            #     continue
            print(f"\nTesting fold: {fold}")

            hdf5_dataset_path = main_hdf5_dataset_path[:-5] + f'_fold{fold}.hdf5'

            if hdf5_path:
                domain_dataloaders = get_domain_dataloaders_from_hdf5(hdf5_path=hdf5_dataset_path, domains=domains, img_path_cols=img_path_cols, num_workers=num_workers, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            else:
                domain_dataloaders = get_domain_dataloaders(train_df, seed=seed, batch_sizes=batch, img_path_cols=img_path_cols, transforms=transform_list, num_workers=num_workers, include_test=test_df, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            
            if joint_training:
                domain_dataloaders = pool_domain_dataloaders(domain_dataloaders)

            print(f"\nTesting: {name} Ablation: {ablation}")

            if ablation == 'nomask':
                setup['ablation'] = 'focus'
            elif ablation == 'onlysoc':
                setup['ablation'] = 'scene'
            elif ablation == 'onlyenv':
                setup['ablation'] = 'focus'
            else:
                setup['ablation'] = False

            if scene_as_label:
                setup['scene'] = 'label'
            dropout_rate=0.3
            model = DualBranchModel(dropout_rate=dropout_rate, setup=setup, branch_norm=branch_norm)
            dual_model = model.to(device)
            trainable_params = [p for p in dual_model.parameters() if p.requires_grad]
            lr=1e-3
            optimizer = torch.optim.Adam(trainable_params, lr=lr)
            buffer = NaiveRehearsalBuffer(buffer_size=buffer_size) if use_buffer else None

            epochs=epochs
            exp_name = f"{name}_{fold=}_{buffer_size=}_{epochs=}_{seed=}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            scheduler=(get_linear_schedule_with_warmup, 0.05)
            refresh_optimiser=False

            dualbranch_kwargs = {
                    'mse_criterion': loss_fn #nn.MSELoss(),
                }

            config = {
                "timestamp": datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
                "model": {
                    "name": str(type(model)),
                    "setup": setup,
                    "branch_norm":branch_norm,
                    "dropout_rate":dropout_rate,
                },
                "buffer": str(type(buffer)),
                "buffer_size": buffer_size,
                "optimizer": str(type(optimizer)),
                "learning_rate": lr,
                "scheduler": {
                    "type": "transformers.get_linear_schedule_with_warmup",
                    "warmup": scheduler[1],
                },
                "device": str(device),
                "loss": str(dualbranch_kwargs['mse_criterion']),
                "joint_training": joint_training, #trained on pooled domains, not continual learning
                "seed": seed,
                "domains_order": domains,
                "train_val_df": train_val_path,
                "test_df": test_path,
                "hdf5_dataset_path": hdf5_dataset_path,
                "input_columns": img_path_cols,
                "input_transforms": [serialize_transform(x) for x in transform_list],
                "dataloader": {
                    "batch_size": batch,
                    "num_workers":num_workers,
                    "pin_memory": pin_memory,
                    "persistent_workers": persistent_workers,
                },
                "epochs": epochs,
                "early_stopping": {
                    "patience":15,
                    "delta":1e-3,
                },
                "gradient_clipping": grad_clipping,
                "refresh_optimiser": refresh_optimiser,
            }

            with open(f"../checkpoints/{exp_name}_config.json", "w") as f:
                json.dump(config, f, indent=4)

            unified_train_loop(
                model=dual_model,
                domains=domains if not joint_training else ['joint'],
                domain_dataloaders=domain_dataloaders,
                buffer=buffer,
                optimizer=optimizer,
                device=device,
                batch_fn=heuristic_dualbranch_batch,
                batch_kwargs=dualbranch_kwargs,
                num_epochs=epochs,
                exp_name=exp_name,
                gradient_clipping=grad_clipping,
                checkpoint_dir="../checkpoints",
                validation_set='val',
                scheduler=scheduler,
                refresh_optimiser=refresh_optimiser,
                early_stopping=earlystopping,
            )